In [4]:
#installing necessary packages

!pip install langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached aiohttp-3.13.3-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached langsmith-0.7.6-py3-none-any.whl.metadata (15 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp312-cp312-win_amd64.whl.metadata (14 kB)
  Using cached yarl-1.22.0-cp312-cp312-win_amd64.whl.metadata (77 kB)
  U


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


<h1>Pre-Processing Steps</h1>

In [7]:
#Create KB by loading docs
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader=DirectoryLoader('kb_files',glob="*.txt",show_progress=True, loader_cls=TextLoader)

kb_docs=loader.load()

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 24.77it/s]


In [56]:
#Create chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=100)

chunks=text_splitter.split_documents(kb_docs)

In [57]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

Number of Documents: 3

Number of Chunks: 5173


In [12]:
!pip install langchain-chroma

  Using cached chromadb-1.5.1-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp312-cp312-win_amd64.whl.metadata (9.1 kB)
  Using cached uvicorn-0.41.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached onnxruntime-1.24.2-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.39.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached grpcio-1.78.1-cp312-cp312-win_amd64.whl.metadata (3.9 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd6


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!pip install langchain-openai

  Using cached tiktoken-0.12.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached regex-2026.2.19-cp312-cp312-win_amd64.whl.metadata (41 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 7.8 MB/s eta 0:00:00
Using cached tiktoken-0.12.0-cp312-cp312-win_amd64.whl (878 kB)
Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl (205 kB)
Using cached regex-2026.2.19-cp312-cp312-win_amd64.whl (277 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
#Add the chunks to vector DB after embedding
from langchain_chroma import Chroma
from langchain_openai import AzureOpenAIEmbeddings
#Get Azure API KEY
f = open("keys/.azure_open_api_key.txt")
AZURE_API_KEY=f.read()
embedding_model=AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    azure_endpoint="https://ai60.cognitiveservices.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2023-05-15",
    openai_api_version="2024-02-01",
    api_key=AZURE_API_KEY
)

In [58]:
#Init DB and add chunked data to db
vdb=Chroma(collection_name="vector_database",
           embedding_function=embedding_model,
           persist_directory="./medico_chroma_db_"
          )
vdb.add_documents(documents=chunks)

['da28e44e-bb5c-4b3a-ae3c-3fa68de7e991',
 'fbf69681-0b67-432d-9c5c-f84c8368e49c',
 'a737343d-86f2-4464-ae7d-ec6dab75522f',
 '576beb2c-a691-4937-81f0-dae7d5fc209f',
 'ab57d81f-7e21-499f-b2df-a791ed1246d7',
 'f2efdc15-07df-46dc-922a-58b280cfc494',
 '3dc418e4-b7e4-4767-b0ef-e78c4d2a41e7',
 '301d7a16-edbb-435d-96ef-9af1d1ff32aa',
 '9b6ef4d2-780a-4504-b770-a7cfaa6f512e',
 '38ed889b-aa7e-42d4-a2a3-4c7c6b24e52a',
 'ef9c4b98-d9f7-4ef9-b74c-0b9f01be13e8',
 '55c8be08-4e95-4be5-85a1-33038f79f200',
 '8e6db783-0fd1-4f04-bdc5-ac75903b83a6',
 '8cf11559-5f47-44c0-8f88-b1013a8388f1',
 '44e1a76d-78ce-4514-9c51-3abe45516486',
 '5264f332-3cd8-40f1-82f8-80db23268891',
 '8fb7d6b0-a1aa-49c7-82dd-c164b34af0eb',
 'fd64b3b3-be3a-4296-b255-2d254f6f5a64',
 '7a9922ab-5d70-4a1c-b01d-347190153a00',
 'fe2bad5a-dfd7-4b97-ac65-6947b753ea26',
 '5cdfa6e2-af79-4df6-a9c4-46bc8fae318b',
 '121cc3b9-371d-4ecc-8830-5977f27269e0',
 'a486cc1a-340d-4027-9c47-21192d6700cd',
 '038177c5-4126-4b40-92c4-2b13d8522cbb',
 '3add5b9d-d9b4-

In [59]:
# We can check the already existing values
print(len(vdb.get()["ids"]))

5452


<h1>Creating the RAG now</h1>

#Here we are doing all in one file but 
#in general these two steps can be different as i understand indexing and RAG creation
1. Get the Db object
2. create the retreiver
3. create the template|llm|parser chaing
4. include the retriever in the chain and start

In [60]:
retriever = vdb.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [52]:
#create the template

from langchain_core.prompts import ChatPromptTemplate

template_prompt="""
You are a professional assistant, who can answer the questions based only on the 
following context {context}
Answer the question based on the above context:{question}
Don`t justify your answers. Don`t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
If a question is asked out of conext, polietly decline saying your dont have
the capability.
Keep your answers concise to one or two sentences only.
"""

prompt_template=ChatPromptTemplate(
    messages=[template_prompt]
)

prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are a professional assistant, who can answer the questions based only on the \nfollowing context {context}\nAnswer the question based on the above context:{question}\nDon`t justify your answers. Don`t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\nIf a question is asked out of conext, polietly decline saying your dont have\nthe capability.\nKeep your answers concise to one or two sentences only.\n'), additional_kwargs={})])

In [41]:
from langchain_openai import AzureChatOpenAI

model=AzureChatOpenAI(
    azure_endpoint="https://ai60.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview",
    api_key=AZURE_API_KEY,
    api_version="2024-02-15-preview",
    deployment_name="gpt-4o-mini",
    temperature=0
)

model


AzureChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002287DE85B50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002287DE84230>, root_client=<openai.lib.azure.AzureOpenAI object at 0x000002287DBD7980>, root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000002287DE86AB0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, disabled_params={'parallel_tool_calls': None}, azure_endpoint='https://ai60.cogniti

In [27]:

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [61]:
generator_chain = prompt_template | model | parser

In [29]:
# Helper function to join the retrieved chunks

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [62]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | format_docs, 
    "question": RunnablePassthrough()
} | generator_chain

In [75]:
query='What is a RAG in the context of AI?'

rag_chain.invoke(query)

"I don't have the capability to answer that question."